In [44]:
import pandas as pd
import soundfile as sf

df = pd.read_csv("./data/timit_synthetic_13.csv")

fs = 16000
audio_l, sr_l = sf.read(df.iloc[26].audio_path)
audio_r, sr_r = sf.read(df.iloc[35].audio_path)
assert fs == sr_l == sr_r

In [48]:
import pyworld as pw
import numpy as np


def zero_pad(audio, target_length):
    assert len(audio) < target_length
    pad_size = target_length - len(audio)
    return np.pad(
        audio,
        pad_width=(pad_size // 2, pad_size - pad_size // 2),
        mode="constant",
        constant_values=0.0,
    )

def get_continuum(audio_l, audio_r, fs=16000):
    if len(audio_l) < len(audio_r):
        audio_l = zero_pad(audio_l, len(audio_r))
    if len(audio_l) > len(audio_r):
        audio_r = zero_pad(audio_r, len(audio_l))

    f0_l, sp_l, ap_l = pw.wav2world(audio_l, fs)
    f0_r, sp_r, ap_r = pw.wav2world(audio_r, fs)

    signals = []
    for alpha in np.linspace(0.0, 1.0, num=11, endpoint=True):
        f0 = f0_l * (1 - alpha) + f0_r * alpha
        sp = sp_l * (1 - alpha) + sp_r * alpha
        ap = ap_l * (1 - alpha) + ap_r * alpha
        signals.append(
            pw.synthesize(f0, sp, ap, fs, pw.default_frame_period)
        )
    return signals

In [50]:
from IPython.display import display, Audio

print("left")
display(Audio(audio_l, rate=fs))
print("right")
display(Audio(audio_r, rate=fs))

for i, signal in enumerate(get_continuum(audio_l, audio_r)):
    print(i)
    display(Audio(signal, rate=fs))

left


right


0


1


2


...

6


7


8


9


10


In [11]:
import sys
!{sys.executable} -m pip install pyworld

  Using cached pyworld-0.3.5.tar.gz (261 kB)
  Installing budone
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp310-cp310-linux_x86_64.whl size=200216 sha256=411e1d5fb3786cbd3520c56837415db333ef58ef50ba6fb0fbb6bb86ba3827d7
  Stored in directory: /home/kwanghec/.cache/pip/wheels/8e/a0/94/52e99161f9460670f11129bff5224ddf1a17915007d8cfa196
Successfully built pyworld
